# Digital Image Processing — Week 5 · Haftalık ödev: CUDA

**Grup: Nmertgules ve Iliya**

Task 1–5'in tamamı CUDA ile yeniden gerçekleniyor

Her piksel işlemi bir **CUDA C++ kernel**'ı ile GPU'da yapılır. Kernel'lar aşağıdaki hücrede düz CUDA C++ olarak yazılmıştır; Python tarafında [CuPy](https://cupy.dev/) bu kaynağı çalışma anında NVRTC ile derler (`cp.RawModule`), görüntüleri GPU belleğine taşır ve kernel'ları başlatır. Sonuçlar her görevde OpenCV'nin CPU çıktısıyla sayısal olarak karşılaştırılır.

| Görev | GPU tarafı |
| --- | --- |
| 1–2 · Kanal histogramları | `hist_kernel`: blok başına paylaşımlı bellekte (shared memory) 256 sayaç, `atomicAdd` ile birleştirme; `bins`, aralık ve maske destekli |
| 3 · Gri dönüşüm + hata | `gray_kernel` + hata değeri (mutlak farklar toplamı) için `sad_kernel` (warp shuffle indirgeme) |
| 4 · Renk uzayları | Her dönüşüm bir `__device__` fonksiyon; `convert_kernel` dönüşümü **fonksiyon pointer** tablosundan (`d_converters`) seçer |
| 5 · X-ray renklendirme | `gray_kernel` → `lut_kernel` (palet paylaşımlı bellekte) |

**Gereksinim:** NVIDIA GPU ve CuPy. Colab'da *Runtime → Change runtime type → T4 GPU* seçmek yeterli; CuPy Colab'da hazır kuruludur.

In [ ]:
import math, subprocess, sys, time
from pathlib import Path
import urllib.request
import cv2
import numpy as np
import matplotlib.pyplot as plt

try:
    import cupy as cp
    cp.cuda.runtime.getDeviceCount()
except Exception as exc:
    raise RuntimeError('Bu notebook NVIDIA GPU ve CuPy ister. Colab: Runtime → Change runtime type → T4 GPU. '
                       f'(Hata: {exc})')

props = cp.cuda.runtime.getDeviceProperties(0)
SM_COUNT = props['multiProcessorCount']
gpu_name = props['name'].decode() if isinstance(props['name'], bytes) else props['name']
print(f'GPU: {gpu_name} · {SM_COUNT} SM · compute capability {props["major"]}.{props["minor"]}')
print(f'CuPy {cp.__version__} · OpenCV {cv2.__version__}')

In [ ]:
from pathlib import Path
import urllib.request
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Repodaki mevcut renkli fotoğrafı kullanıyoruz. Repo kökü çalışma klasöründen yukarı doğru aranır;
# böylece notebook hem repo kökünden hem de week5 içinden çalışır. Repo yoksa (ör. Colab) aynı
# fotoğraf GitHub'dan indirilir.
IMAGE_URL = 'https://raw.githubusercontent.com/Nmertgules/DEEP/main/week4/assets/rocket.jpg'

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / 'week4' / 'assets' / 'rocket.jpg').is_file():
            return folder
    return None

ROOT = find_repo_root()
if ROOT is None:
    ROOT = Path('DEEP').resolve()
    (ROOT / 'week4' / 'assets').mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(IMAGE_URL, ROOT / 'week4' / 'assets' / 'rocket.jpg')
HERE = ROOT / 'week5'
IMAGE = ROOT / 'week4' / 'assets' / 'rocket.jpg'
OUT = HERE / 'results'
OUT.mkdir(parents=True, exist_ok=True)

bgr = cv2.imread(str(IMAGE), cv2.IMREAD_COLOR)
if bgr is None:
    raise FileNotFoundError(f'Görüntü okunamadı: {IMAGE}')
print(f'Görüntü: {bgr.shape[1]} × {bgr.shape[0]}, {bgr.dtype}')

## CUDA C++ kaynağı

Her kernel *grid-stride* döngüsü kullanır: bir thread önce kendi indisindeki pikseli işler, sonra toplam thread sayısı kadar ilerler. Böylece her boyuttaki görüntü, sabit sayıda blokla işlenir.

Task 4'teki dönüşümler `PixelFn` türünde (`void (*)(const uchar*, uchar*)`) fonksiyon pointer'larıdır. `convert_kernel` hangi dönüşümü yaptığını bilmez; `which` indisiyle `d_converters` tablosundan aldığı pointer'ı çağırır. Bu, ödevde C++ için istenen fonksiyon pointer yapısının GPU'daki karşılığıdır.

In [ ]:
CUDA_SRC = r'''
// ---------------------------------------------------------------------------
// Week 5 CUDA kernels. Every pixel is processed by one GPU thread
// (grid-stride loops, so any grid size covers any image size).
// ---------------------------------------------------------------------------
typedef unsigned char uchar;

__device__ __forceinline__ uchar sat_u8(int v) { return (uchar)min(max(v, 0), 255); }
__device__ __forceinline__ int descale(int x, int n) { return (x + (1 << (n - 1))) >> n; }

// ----- Task 3: BGR -> GRAY (OpenCV's 15-bit fixed-point BT.601 weights) ------
__device__ void px_bgr2gray(const uchar* s, uchar* d) {
    d[0] = (uchar)((s[0] * 3735 + s[1] * 19235 + s[2] * 9798 + (1 << 14)) >> 15);
}

// ----- Task 4: colour-space conversions, one pixel each ---------------------
__device__ __forceinline__ int luma14(int b, int g, int r) {
    return descale(b * 1868 + g * 9617 + r * 4899, 14);
}

__device__ void px_bgr2ycrcb(const uchar* s, uchar* d) {
    int b = s[0], g = s[1], r = s[2], y = luma14(b, g, r);
    d[0] = sat_u8(y);
    d[1] = sat_u8(descale((r - y) * 11682 + (128 << 14), 14));
    d[2] = sat_u8(descale((b - y) * 9241 + (128 << 14), 14));
}

__device__ void px_bgr2yuv(const uchar* s, uchar* d) {
    int b = s[0], g = s[1], r = s[2], y = luma14(b, g, r);
    d[0] = sat_u8(y);
    d[1] = sat_u8(descale((b - y) * 8061 + (128 << 14), 14));
    d[2] = sat_u8(descale((r - y) * 14369 + (128 << 14), 14));
}

__device__ void px_bgr2xyz(const uchar* s, uchar* d) {
    // sRGB/D65 matrix, coefficients round(c * 4096)
    int b = s[0], g = s[1], r = s[2];
    d[0] = sat_u8(descale(r * 1689 + g * 1465 + b * 739, 12));
    d[1] = sat_u8(descale(r * 871 + g * 2929 + b * 296, 12));
    d[2] = sat_u8(descale(r * 79 + g * 488 + b * 3892, 12));
}

__device__ void px_bgr2hsv(const uchar* s, uchar* d) {
    const int shift = 12;
    int b = s[0], g = s[1], r = s[2];
    int v = max(max(b, g), r);
    int diff = v - min(min(b, g), r);
    // Same division tables as OpenCV: round(255*2^12 / v) and round(180*2^12 / (6*diff)),
    // rounded half-to-even like cvRound.
    int sdiv = v ? __double2int_rn((double)(255 << shift) / v) : 0;
    int hdiv = diff ? __double2int_rn((double)(180 << shift) / (6.0 * diff)) : 0;
    int sat = descale(diff * sdiv, shift);
    int h = (v == r) ? g - b : (v == g) ? b - r + 2 * diff : r - g + 4 * diff;
    h = descale(h * hdiv, shift);
    if (h < 0) h += 180;
    d[0] = sat_u8(h);
    d[1] = sat_u8(sat);
    d[2] = (uchar)v;
}

__device__ void px_ycrcb2bgr(const uchar* s, uchar* d) {
    int y = s[0], cr = s[1] - 128, cb = s[2] - 128;
    d[0] = sat_u8(y + descale(cb * 29049, 14));
    d[1] = sat_u8(y + descale(cb * -5636 + cr * -11698, 14));
    d[2] = sat_u8(y + descale(cr * 22987, 14));
}

// Function-pointer table: the kernel does not know which conversion it runs.
typedef void (*PixelFn)(const uchar*, uchar*);
__device__ PixelFn d_converters[] = {
    px_bgr2gray, px_bgr2ycrcb, px_bgr2yuv, px_bgr2xyz, px_bgr2hsv, px_ycrcb2bgr
};

// ------------------------------ kernels ------------------------------

// Tasks 1 & 2: histogram of one channel of an interleaved image.
// Per-block histogram in shared memory, then one atomicAdd per bin to global memory.
extern "C" __global__ void hist_kernel(const uchar* img, int n_pixels, int n_channels, int channel,
                                       int bins, int low, int high, const uchar* mask, int use_mask,
                                       unsigned int* hist) {
    __shared__ unsigned int local[256];
    for (int i = threadIdx.x; i < bins; i += blockDim.x) local[i] = 0;
    __syncthreads();
    int width = high - low;
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n_pixels; i += gridDim.x * blockDim.x) {
        if (use_mask && mask[i] == 0) continue;
        int v = img[(size_t)i * n_channels + channel];
        if (v >= low && v < high) atomicAdd(&local[(v - low) * bins / width], 1u);
    }
    __syncthreads();
    for (int i = threadIdx.x; i < bins; i += blockDim.x)
        if (local[i]) atomicAdd(&hist[i], local[i]);
}

// Task 3: direct grayscale kernel.
extern "C" __global__ void gray_kernel(const uchar* bgr, uchar* gray, int n_pixels) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n_pixels; i += gridDim.x * blockDim.x)
        px_bgr2gray(bgr + (size_t)i * 3, gray + i);
}

// Task 4: generic conversion kernel, the conversion is chosen through a function pointer.
extern "C" __global__ void convert_kernel(const uchar* src, uchar* dst, int n_pixels,
                                          int which, int out_channels) {
    PixelFn fn = d_converters[which];
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n_pixels; i += gridDim.x * blockDim.x)
        fn(src + (size_t)i * 3, dst + (size_t)i * out_channels);
}

// Error value: sum of absolute differences, warp shuffle reduction + one atomic per warp.
extern "C" __global__ void sad_kernel(const uchar* a, const uchar* b, unsigned long long n,
                                      unsigned long long* out) {
    unsigned long long acc = 0;
    for (unsigned long long i = blockIdx.x * (unsigned long long)blockDim.x + threadIdx.x; i < n;
         i += (unsigned long long)gridDim.x * blockDim.x)
        acc += (unsigned long long)abs((int)a[i] - (int)b[i]);
    for (int offset = 16; offset > 0; offset >>= 1)
        acc += __shfl_down_sync(0xffffffffu, acc, offset);
    if ((threadIdx.x & 31) == 0 && acc) atomicAdd(out, acc);
}

// Task 5: palette lookup (256 x 3 BGR table kept in shared memory).
extern "C" __global__ void lut_kernel(const uchar* gray, const uchar* lut, uchar* out, int n_pixels) {
    __shared__ uchar table[256 * 3];
    for (int i = threadIdx.x; i < 256 * 3; i += blockDim.x) table[i] = lut[i];
    __syncthreads();
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < n_pixels; i += gridDim.x * blockDim.x) {
        const uchar* c = table + gray[i] * 3;
        uchar* o = out + (size_t)i * 3;
        o[0] = c[0]; o[1] = c[1]; o[2] = c[2];
    }
}
'''

module = cp.RawModule(code=CUDA_SRC, options=('--std=c++14',))
hist_kernel = module.get_function('hist_kernel')
gray_kernel = module.get_function('gray_kernel')
convert_kernel = module.get_function('convert_kernel')
sad_kernel = module.get_function('sad_kernel')
lut_kernel = module.get_function('lut_kernel')

BLOCK = 256

def grid_for(n):
    """n eleman için blok sayısı; SM başına en çok 32 blok (fazlası grid-stride döngüsüyle işlenir)."""
    return (max(1, min(math.ceil(n / BLOCK), SM_COUNT * 32)),)

def to_gpu(arr):
    return cp.ascontiguousarray(cp.asarray(arr))

def gpu_sad(a, b):
    """İki GPU dizisinin mutlak farklarının toplamı (hata değeri), GPU'da hesaplanır."""
    a, b = to_gpu(a), to_gpu(b)
    assert a.shape == b.shape and a.dtype == b.dtype == cp.uint8
    out = cp.zeros(1, dtype=cp.uint64)
    sad_kernel(grid_for(a.size), (BLOCK,), (a, b, np.uint64(a.size), out))
    return int(out.get()[0])

print('CUDA kaynağı derlendi: hist, gray, convert, sad, lut kernel\'ları hazır.')

## Task 1 & 2 · Kanal histogramları

`gpu_calc_hist`, Task 2'deki `calc_hist` ile aynı arayüze sahiptir (`bins`, `value_range`, `mask`). Kanallar ayrı ayrı hesaplanır; görüntü GPU'ya bir kez yüklenir ve kernel'a hangi kanalın sayılacağı `channel` parametresiyle verilir. Referans, Task 1'deki gibi `cv2.calcHist`'tir.

In [ ]:
def gpu_calc_hist(img_gpu, channel=0, bins=256, value_range=(0, 256), mask=None):
    low, high = value_range
    if not (1 <= bins <= 256) or low >= high:
        raise ValueError('1 <= bins <= 256 ve alt < üst olmalı.')
    n_channels = 1 if img_gpu.ndim == 2 else img_gpu.shape[2]
    n_pixels = img_gpu.shape[0] * img_gpu.shape[1]
    hist = cp.zeros(bins, dtype=cp.uint32)
    mask_gpu = to_gpu(mask) if mask is not None else cp.zeros(1, dtype=cp.uint8)
    hist_kernel(grid_for(n_pixels), (BLOCK,),
                (img_gpu, np.int32(n_pixels), np.int32(n_channels), np.int32(channel),
                 np.int32(bins), np.int32(low), np.int32(high), mask_gpu, np.int32(mask is not None), hist))
    return hist.get().astype(np.int64)

bgr_gpu = to_gpu(bgr)
names, colors = ['Blue (B)', 'Green (G)', 'Red (R)'], ['blue', 'green', 'red']
gpu_hists = [gpu_calc_hist(bgr_gpu, c) for c in range(3)]
for name, c, h in zip(names, range(3), gpu_hists):
    ref = cv2.calcHist([bgr], [c], None, [256], [0, 256]).ravel().astype(np.int64)
    print(f'{name}: toplam {h.sum()} piksel · cv2.calcHist ile maksimum fark = {np.abs(h - ref).max()}')
    assert np.array_equal(h, ref)

# Parametreli kontroller (Task 2'deki gibi OpenCV'nin de kesin olduğu aralıklar)
H, W = bgr.shape[:2]
yy, xx = np.ogrid[:H, :W]
circle = (((yy - H / 2) ** 2 + (xx - W / 2) ** 2) <= (min(H, W) / 3) ** 2).astype(np.uint8) * 255
for bins, rng, mask in [(16, (0, 256), None), (64, (0, 256), None), (8, (64, 192), None),
                        (256, (0, 256), circle), (32, (0, 256), circle), (4, (128, 256), circle)]:
    for c in range(3):
        ref = cv2.calcHist([bgr], [c], mask, [bins], list(rng)).ravel().astype(np.int64)
        assert np.array_equal(gpu_calc_hist(bgr_gpu, c, bins, rng, mask), ref)
print('Parametre kontrolleri (bins, aralık, dairesel maske) geçti.')

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, h, name, color in zip(axes, gpu_hists, names, colors):
    ax.plot(np.arange(256), h, color=color)
    ax.set(title=f'{name} · CUDA', xlabel='Pixel value', ylabel='Pixel count', xlim=(0, 255))
    ax.grid(alpha=0.2)
fig.tight_layout(); fig.savefig(OUT / 'cuda_task1_2_histograms.svg'); plt.show()

## Task 3 · Gri dönüşüm ve hata değeri

`gray_kernel`, Task 3'teki sabit noktalı BT.601 formülünü (`(3735·B + 19235·G + 9798·R + 2¹⁴) >> 15`) her thread'de bir piksel için uygular. Hata değeri de GPU'da `sad_kernel` ile hesaplanır.

In [ ]:
def gpu_bgr_to_gray(img_gpu):
    n = img_gpu.shape[0] * img_gpu.shape[1]
    gray = cp.empty(img_gpu.shape[:2], dtype=cp.uint8)
    gray_kernel(grid_for(n), (BLOCK,), (img_gpu, gray, np.int32(n)))
    return gray

gray_gpu = gpu_bgr_to_gray(bgr_gpu)
gray_cv = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
print(f'Boyut: {gray_gpu.shape}, tek kanal, {gray_gpu.dtype}')
print(f'HATA DEĞERİ (OpenCV vs CUDA gri): {gpu_sad(gray_gpu, gray_cv)}')

# 2^24 = 16.777.216 olası 8 bit BGR rengin tamamı, 4096 × 4096 boyutlu tek bir görüntüde.
codes = np.arange(1 << 24, dtype=np.uint32)
all_colors = np.dstack([codes & 255, (codes >> 8) & 255, (codes >> 16) & 255]).astype(np.uint8).reshape(4096, 4096, 3)
all_gpu = to_gpu(all_colors)
full = gpu_sad(gpu_bgr_to_gray(all_gpu), cv2.cvtColor(all_colors, cv2.COLOR_BGR2GRAY))
print(f'16.777.216 rengin tamamında hata değeri: {full}')
assert full == 0

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title('Original')
axes[1].imshow(gray_gpu.get(), cmap='gray', vmin=0, vmax=255); axes[1].set_title('Gray · CUDA')
for ax in axes: ax.axis('off')
fig.tight_layout(); plt.show()

## Task 4 · Renk uzayı dönüşümleri (fonksiyon pointer'ları)

`CONVERSIONS` listesinin sırası, CUDA tarafındaki `d_converters` pointer tablosunun sırasıyla aynıdır. Python yalnızca indisi gönderir; GPU'da hangi fonksiyonun çalışacağını pointer belirler.

In [ ]:
CONVERSIONS = [  # (ad, d_converters indisi, çıkış kanal sayısı, OpenCV kodu)
    ('BGR → GRAY',  0, 1, cv2.COLOR_BGR2GRAY),
    ('BGR → YCrCb', 1, 3, cv2.COLOR_BGR2YCrCb),
    ('BGR → YUV',   2, 3, cv2.COLOR_BGR2YUV),
    ('BGR → XYZ',   3, 3, cv2.COLOR_BGR2XYZ),
    ('BGR → HSV',   4, 3, cv2.COLOR_BGR2HSV),
    ('YCrCb → BGR', 5, 3, cv2.COLOR_YCrCb2BGR),
]

def gpu_convert(img_gpu, which, out_channels):
    h, w = img_gpu.shape[:2]
    out = cp.empty((h, w) if out_channels == 1 else (h, w, out_channels), dtype=cp.uint8)
    convert_kernel(grid_for(h * w), (BLOCK,), (img_gpu, out, np.int32(h * w), np.int32(which), np.int32(out_channels)))
    return out

for label, image in [('fotoğraf', bgr), ('16.777.216 renk', all_colors)]:
    print(f'\n{label}:')
    print(f'  {"Dönüşüm":<13} {"Hata (SAD)":>10}')
    for name, which, ch, code_ in CONVERSIONS:
        source = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb) if name.startswith('YCrCb') else image
        ours = gpu_convert(to_gpu(source), which, ch)
        err = gpu_sad(ours, cv2.cvtColor(source, code_))
        print(f'  {name:<13} {err:>10}')
        assert err == 0, name
print('\nBütün dönüşümlerde hata değeri 0: beklenen sonuç sağlandı.')

## Task 5 · X-ray görüntülerinin GPU'da renklendirilmesi

Task 5'teki 5 görüntü ve aynı palet kullanılır. Palet (256 renklik tablo) CPU'da bir kez hesaplanır; piksel başına işler (griye çevirme ve tablodan renk okuma) GPU'da yapılır. Sonuç, CPU sürümüyle birebir karşılaştırılır.

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # kagglehub ilerleme çubuğu uyarısı
try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
    import kagglehub

DATASET = Path(kagglehub.dataset_download('orvile/hums-x-ray-dataset'))
THREATS = next(p for p in DATASET.rglob('ThreatsRGB') if p.is_dir())
SELECTED = ['image_0_1072.jpg', 'image_0_1212.jpg', 'image_0_1302.jpg', 'image_0_1480.jpg', 'image_0_1922.jpg']
originals = [cv2.imread(str(THREATS / name), cv2.IMREAD_COLOR) for name in SELECTED]

# Task 5 ile aynı palet: L* gri seviyeden, a*/b* çapa noktalarından.
ANCHORS = [(0, (25, -55)), (110, (5, -60)), (142, (-10, -40)), (158, (-45, 40)),
           (175, (35, 60)), (212, (25, 55)), (224, (2, 38)), (234, (-12, 32)), (246, (-4, 10)), (255, (0, 0))]

def build_palette(anchors):
    levels = np.arange(256, dtype=np.uint8)
    as_bgr = cv2.cvtColor(levels.reshape(1, -1), cv2.COLOR_GRAY2BGR).astype(np.float32) / 255
    L = cv2.cvtColor(as_bgr, cv2.COLOR_BGR2Lab)[0, :, 0]
    xs = [g for g, _ in anchors]
    a = np.interp(levels, xs, [ab[0] for _, ab in anchors])
    b = np.interp(levels, xs, [ab[1] for _, ab in anchors])
    bgr_float = cv2.cvtColor(np.dstack([L, a, b]).astype(np.float32), cv2.COLOR_Lab2BGR)[0]
    return np.clip(np.rint(bgr_float * 255), 0, 255).astype(np.uint8)

PALETTE = build_palette(ANCHORS)
palette_gpu = to_gpu(PALETTE)

def gpu_colorize(img_gpu):
    gray = gpu_bgr_to_gray(img_gpu)
    h, w = gray.shape
    out = cp.empty((h, w, 3), dtype=cp.uint8)
    lut_kernel(grid_for(h * w), (BLOCK,), (gray, palette_gpu, out, np.int32(h * w)))
    return gray, out

results = []
for name, im in zip(SELECTED, originals):
    gray_g, col_g = gpu_colorize(to_gpu(im))
    cpu_col = PALETTE[cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)]
    err = gpu_sad(col_g, cpu_col)
    print(f'{name}: CUDA renklendirme vs CPU (Task 5) hata değeri = {err}')
    assert err == 0
    results.append((gray_g.get(), col_g.get()))

rgb = lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(len(SELECTED), 3, figsize=(13, 3.3 * len(SELECTED)))
for row, name, im, (g, c) in zip(axes, SELECTED, originals, results):
    row[0].imshow(rgb(im)); row[0].set_title(f'{name} · original')
    row[1].imshow(g, cmap='gray', vmin=0, vmax=255); row[1].set_title('gray · CUDA')
    row[2].imshow(rgb(c)); row[2].set_title('palette · CUDA')
    for ax in row: ax.axis('off')
fig.tight_layout(); fig.savefig(OUT / 'cuda_task5_xray.png', dpi=70); plt.show()

## Süre karşılaştırması

16.777.216 piksellik (4096 × 4096) görüntüde OpenCV'nin CPU fonksiyonları ile CUDA kernel'larının süreleri. GPU süresi CUDA event'leriyle yalnızca kernel için ölçülür; görüntünün GPU'ya kopyalanma süresi ayrıca verilir. Sonuçlar donanıma göre değişir.

In [ ]:
def cpu_ms(fn, repeat=5):
    fn()
    t = time.perf_counter()
    for _ in range(repeat):
        fn()
    return (time.perf_counter() - t) * 1000 / repeat

def gpu_ms(fn, repeat=20):
    fn(); cp.cuda.Device().synchronize()
    start, end = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    for _ in range(repeat):
        fn()
    end.record(); end.synchronize()
    return cp.cuda.get_elapsed_time(start, end) / repeat

upload = gpu_ms(lambda: cp.asarray(all_colors), repeat=5)
rows = [
    ('Histogram (B kanalı)', cpu_ms(lambda: cv2.calcHist([all_colors], [0], None, [256], [0, 256])),
                             gpu_ms(lambda: gpu_calc_hist(all_gpu, 0))),
    ('BGR → GRAY',           cpu_ms(lambda: cv2.cvtColor(all_colors, cv2.COLOR_BGR2GRAY)),
                             gpu_ms(lambda: gpu_bgr_to_gray(all_gpu))),
    ('BGR → HSV (pointer)',  cpu_ms(lambda: cv2.cvtColor(all_colors, cv2.COLOR_BGR2HSV)),
                             gpu_ms(lambda: gpu_convert(all_gpu, 4, 3))),
    ('Gri → palet (LUT)',    cpu_ms(lambda: cv2.applyColorMap(cv2.cvtColor(all_colors, cv2.COLOR_BGR2GRAY), PALETTE.reshape(256, 1, 3))),
                             gpu_ms(lambda: gpu_colorize(all_gpu))),
]
print(f'GPU: {gpu_name} · görüntünün CPU → GPU kopyalanması: {upload:.1f} ms')
print(f'{"İşlem":<22} {"OpenCV CPU (ms)":>16} {"CUDA (ms)":>10}')
for name, c, g in rows:
    print(f'{name:<22} {c:16.2f} {g:10.2f}')
print('Not: histogram satırındaki CUDA süresi sonucun CPU\'ya kopyalanmasını da içerir.')

## Sonuç

- **Task 1–2:** CUDA histogramı her kanalda `cv2.calcHist` ile birebir aynı; farklı kutu sayıları, aralıklar ve maske ile de aynı.
- **Task 3:** CUDA gri dönüşümünün OpenCV'ye göre hata değeri fotoğrafta ve bütün 16,7 milyon renkte **0**.
- **Task 4:** Fonksiyon pointer tablosuyla seçilen altı dönüşümün hepsinde hata değeri **0** (beklenen değer).
- **Task 5:** Griye çevirme ve paletle renklendirme GPU'da yapıldı; CPU sonucuyla hata değeri **0**. Paletin gerekçesi ve orijinalle farkların yorumu `05_xray_pseudocolor.ipynb` içinde.

GPU, piksel başına bağımsız işlerde (dönüşüm, LUT) büyük görüntülerde belirgin hız kazandırır. Küçük görüntülerde ise veri kopyalama ve kernel başlatma maliyeti kazancı azaltabilir. Fonksiyon pointer'ı ile dolaylı çağrı derleyicinin fonksiyonu satır içine almasını (inline) engeller; bu esneklik küçük bir hız maliyetiyle gelir.